# Driver Pickup ETA Prediction -- Data Acquisition

## 1.1  What Problem Are We Solving?

When a rider opens Uber or Lyft and taps **Request Ride**, the app immediately shows:
**'Your driver will arrive in X minutes.'**
That number is the **Pickup ETA** -- one of the most consequential predictions the platform makes.

**Why it matters for the business:**
- Accurate ETAs build rider trust and reduce cancellations (lost revenue)
- Bad ETAs cause pickup mismatches -- rider walks to wrong corner, driver circles
- Dispatch algorithms rely on ETA to match riders with the *closest* available driver
- Surge pricing models use supply/demand signals that depend on accurate ETA estimates

**What we will build:** A regression model that predicts `eta_seconds` -- the number
of seconds from when a rider submits a trip request to when the driver physically
arrives at the pickup location.

## 1.2  Why NYC TLC HVFHV Data (Not a Kaggle Dataset)?

The **New York City Taxi and Limousine Commission (TLC)** mandates by law that all
High Volume For-Hire Vehicle (HVFHV) operators -- including **Uber (HV0003)** and
**Lyft (HV0005)** -- submit every trip record to a public database.

This is raw, production-grade, legally mandated data with real-world noise:
- GPS failures causing null coordinates
- Drivers forgetting to tap 'arrived' causing missing `on_scene_datetime`
- App crashes creating corrupted timestamps
- Ghost trips (0 miles, 0 seconds)
- Negative ETAs from out-of-order timestamps

Working with this data forces the same engineering decisions a **production ML engineer
at Uber** would face -- which is exactly why this is a portfolio-worthy project.

**Dataset:** HVFHV Trip Data, January 2024  
**Source:** https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page  
**Format:** Apache Parquet (~20 million trips per month)


In [ ]:
# Cell 2: Imports and logging setup
# We configure Python's built-in logging so every step produces timestamped output.
# This is essential for debugging long-running data downloads.

import os, sys, logging
from pathlib import Path

import numpy as np
import pandas as pd
import requests

try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

sys.path.insert(0, str(Path.cwd().parent))
import config

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger(__name__)

RANDOM_STATE = config.RANDOM_STATE
logger.info('Libraries loaded. Random state = %d', RANDOM_STATE)
print(f'Pandas {pd.__version__} | NumPy {np.__version__}')


## 1.3  Why Parquet? Why Not CSV?

| Feature | CSV | Parquet |
|---|---|---|
| Storage format | Row-oriented | **Column-oriented** |
| Compression | None by default | Snappy/GZIP built-in (~3x smaller) |
| Schema | No types -- everything is text | **Types enforced** (datetime, int, float) |
| Read speed | Slow -- reads all columns | **Fast -- reads only requested columns** |
| Null handling | Ambiguous (empty string vs null) | Explicit null representation |
| Industry use | Legacy data exchange | Spark, BigQuery, Snowflake, Delta Lake |

**Columnar storage** means if you need 5 of 30 columns, Parquet reads only those 5.
For a 20M-row dataset this is the difference between a 5-second and 2-minute load.

**Schema enforcement** means `request_datetime` is stored as a proper datetime object,
not the string `'2024-01-15 08:32:11'` -- eliminating an entire class of parsing bugs.


In [ ]:
# Cell 4: Download the HVFHV Parquet file
# Uses streaming mode so we never load the entire file into RAM during download.
# Idempotent: if the file already exists on disk, we skip the download.

RAW_DIR  = Path('..') / config.RAW_DATA_PATH
RAW_DIR.mkdir(parents=True, exist_ok=True)
RAW_FILE = RAW_DIR / config.RAW_FILE_NAME

if RAW_FILE.exists():
    size_mb = RAW_FILE.stat().st_size / (1024 ** 2)
    logger.info('File already exists: %s (%.1f MB) -- skipping download.', RAW_FILE, size_mb)
else:
    logger.info('Starting download: %s', config.HVFHV_URL)
    resp = requests.get(config.HVFHV_URL, stream=True, timeout=180)
    resp.raise_for_status()
    total = int(resp.headers.get('content-length', 0))
    with open(RAW_FILE, 'wb') as f:
        with tqdm(total=total, unit='iB', unit_scale=True,
                  unit_divisor=1024, desc='Downloading') as bar:
            for chunk in resp.iter_content(chunk_size=8192):
                bar.update(f.write(chunk))

size_mb = RAW_FILE.stat().st_size / (1024 ** 2)
print(f'Raw file ready: {RAW_FILE.name}  ({size_mb:.1f} MB)')


## 1.5  First Look (Why We Sample, Not Load Everything)

The full HVFHV January 2024 dataset has ~20 million rows.
Loading all rows into pandas on a laptop requires ~10 GB RAM and makes
even `.describe()` take minutes.

**Strategy:** Sample **500,000 rows** (`SAMPLE_SIZE` from `config.py`) with
`random_state=42`. This gives us:
- Statistically stable distributional estimates for all features
- Coverage of all 260+ taxi zones, both operators, all 24 hours
- Sub-second pandas operations for fast iteration

**In production** this pipeline runs on Apache Spark, Google BigQuery, or AWS EMR.
Sampling is an *engineering decision*, not a shortcut -- it is documented and reproducible.


In [ ]:
# Cell 6: Load parquet, sample 500k rows, inspect
SAMPLE_SIZE = config.SAMPLE_SIZE
INTERIM_DIR = Path('..') / config.INTERIM_DATA_PATH
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

logger.info('Loading parquet...')
df_full = pd.read_parquet(RAW_FILE)
logger.info('Full dataset: %d rows x %d cols', *df_full.shape)

df = df_full.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE).reset_index(drop=True)

mem_full = df_full.memory_usage(deep=True).sum() / 1024**2
mem_samp = df.memory_usage(deep=True).sum() / 1024**2

print(f'Full dataset:   {df_full.shape[0]:>12,} rows | {mem_full:.1f} MB')
print(f'Sample:         {df.shape[0]:>12,} rows | {mem_samp:.1f} MB')
print()
print('--- dtypes ---')
print(df.dtypes.to_string())
print()
print('--- head(5) ---')
display(df.head())
print()
print('--- tail(5) ---')
display(df.tail())

sample_path = INTERIM_DIR / config.INTERIM_SAMPLE_NAME
df.to_parquet(sample_path, index=False)
logger.info('Sample saved -> %s', sample_path)


## 1.7  Column Dictionary

| Column | Type | Meaning | Safe as feature? |
|---|---|---|---|
| `hvfhs_license_num` | str | Company: HV0003=Uber, HV0005=Lyft | YES |
| `request_datetime` | datetime | When rider submitted request -- our **time-zero** | YES (derive features) |
| `on_scene_datetime` | datetime | When driver physically arrived -- **target endpoint** | NO (constructs target) |
| `pickup_datetime` | datetime | When trip started (after on_scene) | NO (post-arrival) |
| `dropoff_datetime` | datetime | When trip ended | NO (post-trip) |
| `PULocationID` | int | TLC taxi zone for pickup (~265 zones) | YES |
| `DOLocationID` | int | TLC taxi zone for dropoff | YES |
| `trip_miles` | float | GPS-measured distance | CAREFUL (see below) |
| `trip_time` | int | Total trip seconds | NO -- POST-TRIP |
| `base_passenger_fare` | float | Fare before extras | NO -- POST-TRIP |
| `driver_pay` | float | Driver earnings | NO -- POST-TRIP |
| `shared_request_flag` | str | Y = rider requested pooled ride | YES |
| `wav_request_flag` | str | Y = wheelchair accessible vehicle | YES |

**Note on `trip_miles`:** The estimated destination distance is sometimes available
at request time (riders enter destination). We treat this carefully and verify it
is not leaking -- in the HVFHV dataset, `trip_miles` reflects actual GPS tracking
during the trip, so it is strictly a post-trip column and must NOT be used as a feature.


In [ ]:
# Cell 8: Target variable construction -- eta_seconds
# This is the most critical cell. Garbage labels produce a garbage model.

for col in ['request_datetime', 'on_scene_datetime']:
    if col in df.columns and df[col].dtype == object:
        df[col] = pd.to_datetime(df[col], errors='coerce')

# Derive target: time from rider request to driver arrival
df['eta_seconds'] = (df['on_scene_datetime'] - df['request_datetime']).dt.total_seconds()

total = len(df)
n_null   = df['on_scene_datetime'].isna().sum()
n_neg    = (df['eta_seconds'] < 0).sum()
n_over1h = (df['eta_seconds'] > 3600).sum()
n_fast   = (df['eta_seconds'] < 30).sum()
n_valid  = df['eta_seconds'].between(30, 3600).sum()

report = pd.DataFrame({
    'Issue': [
        'on_scene_datetime is null',
        'eta_seconds negative (timestamp error)',
        'eta_seconds > 3600 s (over 1 hour)',
        'eta_seconds < 30 s (suspiciously fast)',
        'VALID: 30 s <= eta <= 3600 s',
    ],
    'Count':  [n_null, n_neg, n_over1h, n_fast, n_valid],
    'Pct':    [f'{100*n/total:.2f}%' for n in [n_null, n_neg, n_over1h, n_fast, n_valid]],
})
print(f'Total rows: {total:,}')
display(report)


## 1.9  MCAR / MAR / MNAR -- Choosing What To Do With Null `on_scene_datetime`

Missing data has three distinct mechanisms -- the mechanism determines the correct action:

| Mechanism | Definition | Safe to impute? |
|---|---|---|
| **MCAR** | Missing Completely At Random -- random hardware glitch | Yes |
| **MAR** | Missing conditionally on observed variables | Sometimes |
| **MNAR** | Missing because of the value itself | Dangerous |

**Our situation is MNAR.** `on_scene_datetime` is null when the driver did NOT tap
'I've arrived' in the app. This happens more often for:
- Very short ETAs (driver was already there -- no need to tap)
- Confusing pickup locations (driver never properly arrived)
- Less app-compliant drivers

The null values **cluster on specific ETA ranges** -- exactly the distribution we are
trying to model. Imputing would introduce systematic bias on the target's extremes.

**Decision: DROP rows where `on_scene_datetime` is null.**

We do NOT impute the target variable because:
> Imputing `eta_seconds` with feature-based predictions means training on labels
> we fabricated from the same features. The model learns a tautology -- artificially
> inflated offline metrics, complete failure in production.
> This is **target imputation leakage** -- one of the most dangerous forms of data leakage.


In [ ]:
# Cell 10: Apply sequential filters -- log rows removed after each step
# Logging each step separately lets us audit the impact of every decision.

df_clean = df.copy()
print(f'Starting rows: {len(df_clean):,}')
print('-' * 55)

steps = [
    ('Drop null on_scene_datetime',
     lambda d: d.dropna(subset=['on_scene_datetime'])),
    ('Remove non-positive eta',
     lambda d: d[d['eta_seconds'] > 0]),
    ('Remove eta > 3600 s',
     lambda d: d[d['eta_seconds'] <= 3600]),
    ('Remove eta < 30 s',
     lambda d: d[d['eta_seconds'] >= 30]),
]

for label, fn in steps:
    before = len(df_clean)
    df_clean = fn(df_clean)
    print(f'{label:<35} -> {len(df_clean):>8,}  (removed {before - len(df_clean):,})')

# Ghost trips: 0 miles AND 0 trip time
if 'trip_time' in df_clean.columns:
    before = len(df_clean)
    ghost = (df_clean['trip_miles'] == 0) & (df_clean['trip_time'] == 0)
    df_clean = df_clean[~ghost]
    print(f'{"Remove ghost trips (0 mi, 0 s)":<35} -> {len(df_clean):>8,}  (removed {before - len(df_clean):,})')

print('-' * 55)
print(f'Final retained: {len(df_clean):,}  ({100*len(df_clean)/len(df):.1f}% of sample)')

df_clean = df_clean.reset_index(drop=True)
out = INTERIM_DIR / config.INTERIM_FILTERED_NAME
df_clean.to_parquet(out, index=False)
print(f'Saved -> {out}')
print()
print('eta_seconds summary:')
print(df_clean['eta_seconds'].describe().round(1))


## 1.11  Notebook 01 Summary

| Step | Action | Output |
|---|---|---|
| Download | Fetched HVFHV Parquet from TLC CDN | `data/raw/fhvhv_2024_01.parquet` |
| Sample | 500k rows, random_state=42 | `data/interim/sample.parquet` |
| Target | Derived `eta_seconds` from timestamps | Column added |
| Audit | Identified null, negative, extreme ETAs | Report printed |
| Filter | 5 sequential quality filters | `data/interim/sample_filtered.parquet` |

**Next:** Notebook 02 loads `sample_filtered.parquet` and performs deep EDA to guide
every feature engineering and modeling decision.

---

## Key Interview Questions -- Notebook 01

**Q: Why sample 500k instead of 10M rows?**  
A: Engineering tradeoff between speed and representativeness. 500k covers all 260 zones,
all hours, both operators. Production uses Spark/BigQuery. Sample size is in `config.py` --
a deliberate, reproducible, documented choice.

**Q: Why Parquet not CSV?**  
A: Columnar (reads only needed columns), compressed 3x, schema-enforced (datetimes stay
datetimes), standard in every modern data platform. CSV at 20M rows would be 5x slower.

**Q: Why drop null `on_scene_datetime` instead of imputing?**  
A: It is our target variable. Imputing it with feature-based predictions creates a tautology
-- the model learns labels it helped generate. Inflated offline metrics, production failure.
Also MNAR: nulls cluster on specific ETA ranges, so imputation would bias the target distribution.

**Q: Explain MCAR vs MAR vs MNAR.**  
A: MCAR = random hardware failure (safe to drop/impute). MAR = missing conditional on observed
variables (can model). MNAR = missing because of the value itself (dangerous -- systematic bias
if imputed). Our case is MNAR -- drivers skip tapping 'arrived' more for unusual trips.

**Q: Why is imputing a target variable dangerous?**  
A: You train on synthetic labels mathematically consistent with features but disconnected from
reality. The model learns a self-referential loop -- excellent CV scores, complete production failure.
